# Prototype: Convert nuScenes JSON to KITTI `.txt` Predictions

This notebook prototypes reading a standard `results.json` generated by a nuScenes evaluation run, mapping the sample tokens to KITTI frame indices, applying the correct coordinate transformations, and exporting `.txt` files that the KITTI evaluator can read.

In [1]:
import json
import pickle
import numpy as np
from pathlib import Path
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import Box
from kittihelper import KittiDB

## 1. Setup Paths and Load Data
Replace `RESULTS_JSON` with the path to your actual nuScenes predictions.

In [2]:
RESULTS_JSON = '/OpenPCDet/output/OpenPCDet/tools/cfgs/custom_models/pointpillar/TENT/pointpillar_nuscenes_TENT/POINTPILLAR_nuscenes_TENT_retrained/eval/epoch_40/test/default/final_result/data/results_nusc.json'
# Use the native OpenPCDet nuScenes infos instead of the converted KITTI DB!
NUSC_INFOS = '/OpenPCDet/datasets/nuscenes/v1.0-mini/nuscenes_infos_10sweeps_val.pkl'  # UPDATE THIS
NUSC_ROOT = '/OpenPCDet/datasets/nuscenes/v1.0-mini'
KITTI_ROOT = '/OpenPCDet/datasets_converted/nuscenes/v1.0-mini_kitti_format'
KITTI_SPLIT = 'mini_val'
OUTPUT_DIR = Path('/OpenPCDet/tools/kitti_style_predictions')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
import os
from pathlib import Path

GT_DIR = '/OpenPCDet/datasets_converted/nuscenes/v1.0-mini_kitti_format/mini_val/label_2'

files = os.listdir(GT_DIR)
print(f"Files in GT_DIR: {len(files)}")
print("Sample files:", files[:5])

Files in GT_DIR: 81
Sample files: ['8573a885a7cb41d185c05029eeb9a54e.txt', 'a19a80c905674faab7203a3a4e0f5246.txt', 'c1eed31234b94e9f8e22fbf3428b0ac2.txt', 'e00dc15130dc44e796687baadd076ae4.txt', '048a45dd2cf54aa5808d8ccc85731d44.txt']


In [4]:
# get_label_annos with explicit file listing to debug
label_files = sorted(Path(GT_DIR).glob('*.txt'))
print(f"Found {len(label_files)} .txt files in GT_DIR")
if label_files:
    print("First file:", label_files[0])
    print("Contents:\n", label_files[0].read_text())

Found 81 .txt files in GT_DIR
First file: /OpenPCDet/datasets_converted/nuscenes/v1.0-mini_kitti_format/mini_val/label_2/048a45dd2cf54aa5808d8ccc85731d44.txt
Contents:
 car 0.00 0 -10.00 1563.28 469.81 1600.00 599.29 1.7 1.74 4.98 13.74 1.39 19.63 1.66 0.0000
motorcycle 0.00 0 -10.00 506.84 516.25 596.34 630.83 1.2 0.51 1.88 -3.15 1.55 15.25 -1.48 0.0000
motorcycle 0.00 0 -10.00 0.00 524.67 80.44 721.01 1.3 0.57 2.19 -6.92 1.62 10.18 -1.49 0.0000
motorcycle 0.00 0 -10.00 869.13 511.19 911.60 589.76 1.2 0.66 1.87 1.24 1.56 21.38 -1.54 0.0000
motorcycle 0.00 0 -10.00 224.09 486.50 383.04 626.97 1.5 0.72 2.61 -6.25 1.50 15.64 1.64 0.0000
car 0.00 0 -10.00 1147.04 497.72 1325.21 614.07 1.5 1.60 4.30 6.44 1.66 19.90 -1.51 0.0000
pedestrian 0.00 0 -10.00 449.40 451.52 569.37 655.61 1.8 0.73 0.82 -2.85 1.44 11.75 2.44 0.0000
motorcycle 0.00 0 -10.00 400.15 496.85 519.49 629.07 1.4 0.67 2.02 -4.19 1.50 15.05 -1.48 0.0000
car 0.00 0 -10.00 1427.62 493.07 1600.00 602.75 1.4 1.71 4.12 11.39 1.52 

In [5]:
PRED_DIR = str(OUTPUT_DIR)
pred_files = sorted(Path(PRED_DIR).glob('*.txt'))
print(f"Found {len(pred_files)} .txt files in PRED_DIR")
if pred_files:
    print("First pred file:", pred_files[0])
    print("Contents:\n", pred_files[0].read_text())

Found 81 .txt files in PRED_DIR
First pred file: /OpenPCDet/tools/kitti_style_predictions/000000.txt
Contents:
 Cyclist 0.00 0 -10.00 0.00 0.00 50.00 50.00 1.6 0.17 1784.56 10.11 4.78 3.92 1.94 1.0000
Cyclist 0.00 0 -10.00 0.00 0.00 50.00 50.00 1.1 0.92 3.53 -48.01 3.64 23.08 0.68 0.9955
Cyclist 0.00 0 -10.00 0.00 0.00 50.00 50.00 2.2 1.88 5.50 19.25 2.69 9.83 0.78 0.9866
Cyclist 0.00 0 -10.00 0.00 0.00 50.00 50.00 1.8 1.02 2.95 -43.78 3.69 15.38 0.37 0.9780
Cyclist 0.00 0 -10.00 0.00 0.00 50.00 50.00 5.5 0.79 1.86 -37.57 4.77 19.04 0.32 0.7912
Cyclist 0.00 0 -10.00 0.00 0.00 50.00 50.00 3.0 1.24 1.95 12.32 3.36 25.51 0.63 0.7311
Cyclist 0.00 0 -10.00 0.00 0.00 50.00 50.00 2.7 1.89 2.14 2.75 3.07 25.25 1.99 0.7271
Cyclist 0.00 0 -10.00 0.00 0.00 50.00 50.00 3.2 0.32 2.65 -8.33 3.38 18.93 1.79 0.6354
Cyclist 0.00 0 -10.00 0.00 0.00 50.00 50.00 2.7 1.08 1.63 -30.44 3.64 23.24 -0.37 0.6305
Cyclist 0.00 0 -10.00 0.00 0.00 50.00 50.00 2.0 0.41 1.99 39.97 1.68 7.00 -0.08 0.5957
Cyclist 0.00 

In [6]:
gt_files   = sorted(Path(GT_DIR).glob('*.txt'))
pred_files = sorted(Path(PRED_DIR).glob('*.txt'))

print("GT   filenames:", [f.name for f in gt_files[:5]])
print("PRED filenames:", [f.name for f in pred_files[:5]])

GT   filenames: ['048a45dd2cf54aa5808d8ccc85731d44.txt', '06be0e3b665c44fa8d17d9f4770bdf9c.txt', '07fad91090c746ccaa1b2bdb55329e20.txt', '0a0d6b8c2e884134a3b48df43d54c36a.txt', '0af0feb5b1394b928dd13d648de898f5.txt']
PRED filenames: ['000000.txt', '000001.txt', '000002.txt', '000003.txt', '000004.txt']


In [7]:
gt_files = sorted(Path(GT_DIR).glob('*.txt'))
pred_files = sorted(Path(PRED_DIR).glob('*.txt'))
gt_stems = {f.stem: f for f in gt_files}
pred_stems = {f.stem: f for f in pred_files}

print("GT stems (first 5):  ", sorted(gt_stems.keys())[:5])
print("PRED stems (first 5):", sorted(pred_stems.keys())[:5])

GT stems (first 5):   ['048a45dd2cf54aa5808d8ccc85731d44', '06be0e3b665c44fa8d17d9f4770bdf9c', '07fad91090c746ccaa1b2bdb55329e20', '0a0d6b8c2e884134a3b48df43d54c36a', '0af0feb5b1394b928dd13d648de898f5']
PRED stems (first 5): ['000000', '000001', '000002', '000003', '000004']


In [8]:
from pcdet.datasets.kitti.kitti_object_eval_python import kitti_common

# Manually load in matched order
gt_files = sorted(Path(GT_DIR).glob('*.txt'))
pred_files = sorted(Path(PRED_DIR).glob('*.txt'))

# Ensure 1-to-1 correspondence by stem
gt_stems = {f.stem: f for f in gt_files}
pred_stems = {f.stem: f for f in pred_files}
common = sorted(set(gt_stems) & set(pred_stems))

print(f"Matched pairs: {len(common)} / GT: {len(gt_stems)} / PRED: {len(pred_stems)}")

image_ids = [int(s) for s in common]
gt_annos = kitti_common.get_label_annos(GT_DIR, image_ids)
dt_annos = kitti_common.get_label_annos(PRED_DIR, image_ids)

Matched pairs: 0 / GT: 81 / PRED: 81


In [9]:
# Load NuScenes dataset context (needed for calibration matrices)
nusc = NuScenes(version='v1.0-mini', dataroot=NUSC_ROOT, verbose=True)

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.4 seconds.
Reverse indexing ...
Done reverse indexing in 0.1 seconds.


## 2. Create Token -> KITTI Matcher
OpenPCDet uses `kitti_infos_val.pkl` to read the converted dataset. We need this to match the `sample_token` from the nuScenes JSON back to the `000000`-style KITTI IDs (often just 0 to N-1 based on the lists).

In [10]:
def get_token_mapping(infos_path):
    """
    Reads the mapping directly from the native OpenPCDet nuScenes infos.
    The order in this list perfectly matches the dataloader indices yielding 000000.txt, 000001.txt, etc.
    """
    with open(infos_path, 'rb') as f:
        infos = pickle.load(f)
    
    token_to_idx = {}
    for idx, info in enumerate(infos):
        # info['token'] is the 32-character nuScenes sample token
        token_to_idx[info['token']] = idx
        
    return token_to_idx

token_to_idx = get_token_mapping(NUSC_INFOS)
print(f"Loaded {len(token_to_idx)} mapping entries directly from nuScenes infos.")

Loaded 81 mapping entries directly from nuScenes infos.


In [11]:
import os

kitti_root = '/OpenPCDet/datasets_converted/nuscenes/v1.0-mini_kitti_format'

for split in ['mini_val', 'val', 'train']:
    split_path = os.path.join(kitti_root, split)
    if os.path.exists(split_path):
        print(f"\n{split}/")
        for folder in os.listdir(split_path):
            folder_path = os.path.join(split_path, folder)
            if os.path.isdir(folder_path):
                n = len(os.listdir(folder_path))
                print(f"  {folder}/  ({n} files)")
            else:
                print(f"  {folder}")


mini_val/
  calib/  (81 files)
  image_2/  (81 files)
  velodyne/  (81 files)
  label_2/  (81 files)


## 3. Coordinate Transformation Mapping
The predictions in the JSON are in the **nuScenes global coordinate frame**. We need to map them back to the **KITTI camera_0 frame**.

In [12]:
KITTI_CLASS_MAP = {
    'car': 'Car',
    'pedestrian': 'Pedestrian',
    'bicycle': 'Cyclist',
}


def map_nusc_class_to_kitti(name: str):
    return KITTI_CLASS_MAP.get(name.lower())

## 4. Iterate over `results_nusc.json`
Assuming we generated the maps and functions successfully, we execute the looping write.

In [13]:
# Reset KittiDB.get_transforms to the true original FIRST before doing anything else.
# Import a fresh reference directly from the module to guarantee it's unwrapped.
import importlib
clean_get_transforms = KittiDB.get_transforms

def process_results(results_path, nusc, token_to_idx, output_dir, kitti_root, kitti_split):
    with open(results_path, 'r') as f:
        data = json.load(f)

    results_dict  = data.get('results', data)
    inv_map       = {v: k for k, v in token_to_idx.items()}  # idx -> sample_token
    empty_frames  = 0
    count         = 0
    skipped_classes = 0

    for sample_token, frame_idx in sorted(token_to_idx.items(), key=lambda kv: kv[1]):

        txt_path = output_dir / f"{frame_idx:06d}.txt"

        # Files on disk are named by sample_token, not zero-padded index.
        # Build the calib path directly instead of going through KittiDB.get_filepath.
        calib_path = Path(kitti_root) / kitti_split / 'calib' / f"{sample_token}.txt"
        if not calib_path.exists():
            raise FileNotFoundError(f"Calib not found: {calib_path}")

        lines       = [line.rstrip() for line in open(calib_path)]
        velo_to_cam = np.array(lines[5].strip().split(' ')[1:], dtype=np.float32).reshape((3, 4))
        r0_rect     = np.array(lines[4].strip().split(' ')[1:], dtype=np.float32).reshape((3, 3))
        p_left      = np.array(lines[2].strip().split(' ')[1:], dtype=np.float32).reshape((3, 4))

        transforms = {
            'velo_to_cam': {'R': velo_to_cam[:, :3], 'T': velo_to_cam[:, 3]},
            'r0_rect':     r0_rect,
            'p_left':      p_left,
        }

        velo_to_cam_rot   = Quaternion(matrix=transforms['velo_to_cam']['R'])
        velo_to_cam_trans = transforms['velo_to_cam']['T']
        r0_rect_q         = Quaternion(matrix=transforms['r0_rect'])

        sample      = nusc.get('sample', sample_token)
        lidar_data  = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
        cs_record   = nusc.get('calibrated_sensor', lidar_data['calibrated_sensor_token'])
        pose_record = nusc.get('ego_pose', lidar_data['ego_pose_token'])

        lines_out   = []
        predictions = results_dict.get(sample_token, [])
        if not predictions:
            empty_frames += 1

        for pred in predictions:
            kitti_class = map_nusc_class_to_kitti(pred['detection_name'])
            if kitti_class is None:
                skipped_classes += 1
                continue

            box_global = Box(
                pred['translation'],
                pred['size'],
                Quaternion(pred['rotation']),
                name=pred['detection_name'],
                score=pred['detection_score']
            )

            # global -> ego -> nuScenes lidar frame
            box_global.translate(-np.array(pose_record['translation']))
            box_global.rotate(Quaternion(pose_record['rotation']).inverse)
            box_global.translate(-np.array(cs_record['translation']))
            box_global.rotate(Quaternion(cs_record['rotation']).inverse)

            box_kitti = KittiDB.box_nuscenes_to_kitti(
                box_global,
                velo_to_cam_rot=velo_to_cam_rot,
                velo_to_cam_trans=velo_to_cam_trans,
                r0_rect=r0_rect_q
            )

            line = KittiDB.box_to_string(
                kitti_class,
                box_kitti,
                bbox_2d=(0.0, 0.0, 50.0, 50.0),
                truncation=0.0,
                occlusion=0,
                alpha=-10.0
            )
            lines_out.append(line + '\n')

        with open(txt_path, 'w') as f:
            f.writelines(lines_out)
        count += 1

    print(f"Exported {count} frames. Empty: {empty_frames}. Skipped classes: {skipped_classes}")


# Make sure KittiDB on the class is also the clean version
KittiDB.get_transforms = clean_get_transforms

process_results(
    RESULTS_JSON,
    nusc,
    token_to_idx,
    OUTPUT_DIR,
    KITTI_ROOT,
    KITTI_SPLIT
)

Exported 81 frames. Empty: 0. Skipped classes: 0


In [14]:
from pcdet.datasets.kitti.kitti_object_eval_python import kitti_common
from pcdet.datasets.kitti.kitti_object_eval_python.eval import get_official_eval_result

GT_DIR   = '/OpenPCDet/datasets_converted/nuscenes/v1.0-mini_kitti_format/mini_val/label_2'
PRED_DIR = str(OUTPUT_DIR)
LOG_FILE = Path('/OpenPCDet/tools/kitti_eval_results_log.txt')

# Build matched GT/pred pairs using token_to_idx for alignment
gt_anno_paths = []
dt_anno_paths = []

for sample_token, frame_idx in sorted(token_to_idx.items(), key=lambda kv: kv[1]):
    gt_path   = Path(GT_DIR)   / f"{sample_token}.txt"  # GT named by sample token
    pred_path = Path(PRED_DIR) / f"{frame_idx:06d}.txt" # preds named by index

    if gt_path.exists() and pred_path.exists():
        gt_anno_paths.append(gt_path)
        dt_anno_paths.append(pred_path)
    else:
        print(f"Missing: GT={gt_path.exists()} PRED={pred_path.exists()} | {sample_token}")

print(f"Aligned pairs: {len(gt_anno_paths)}")
assert len(gt_anno_paths) > 0, "No matched pairs found — check GT_DIR and PRED_DIR"
assert len(gt_anno_paths) == len(dt_anno_paths)

# *** THIS IS THE MISSING STEP — actually load the annos ***
gt_annos = [kitti_common.get_label_anno(str(p)) for p in gt_anno_paths]
dt_annos = [kitti_common.get_label_anno(str(p)) for p in dt_anno_paths]

print(f"GT annos loaded:   {len(gt_annos)}")
print(f"PRED annos loaded: {len(dt_annos)}")

# Quick sanity check on first pair
print(f"\nFirst GT  names: {gt_annos[0]['name']}")
print(f"First DT  names: {dt_annos[0]['name']}")

# Run evaluation
print("\n--- Car Evaluation ---")
car_result = get_official_eval_result(gt_annos, dt_annos, 0)
print(car_result[0] if isinstance(car_result, tuple) else car_result)

with open(LOG_FILE, 'w') as f:
    f.write('--- Car Evaluation ---\n')
    f.write((car_result[0] if isinstance(car_result, tuple) else str(car_result)) + '\n')

    ped_result = get_official_eval_result(gt_annos, dt_annos, 1)
    f.write('\n--- Pedestrian ---\n')
    f.write((ped_result[0] if isinstance(ped_result, tuple) else str(ped_result)) + '\n')

    cyc_result = get_official_eval_result(gt_annos, dt_annos, 2)
    f.write('\n--- Cyclist ---\n')
    f.write((cyc_result[0] if isinstance(cyc_result, tuple) else str(cyc_result)) + '\n')

print(f"\nResults saved to {LOG_FILE}")

Aligned pairs: 81
GT annos loaded:   81
PRED annos loaded: 81

First GT  names: ['pedestrian' 'pedestrian' 'pedestrian' 'car' 'pedestrian' 'pedestrian'
 'pedestrian' 'pedestrian' 'pedestrian' 'car' 'pedestrian' 'pedestrian'
 'pedestrian' 'pedestrian']
First DT  names: ['Pedestrian' 'Cyclist' 'Cyclist' 'Pedestrian' 'Cyclist' 'Cyclist'
 'Cyclist' 'Pedestrian' 'Cyclist' 'Pedestrian' 'Pedestrian' 'Cyclist'
 'Pedestrian' 'Cyclist' 'Pedestrian' 'Cyclist' 'Car' 'Pedestrian'
 'Pedestrian' 'Cyclist' 'Car' 'Cyclist' 'Pedestrian' 'Car' 'Pedestrian'
 'Pedestrian' 'Cyclist' 'Pedestrian' 'Car' 'Pedestrian' 'Pedestrian' 'Car'
 'Car' 'Pedestrian' 'Cyclist' 'Car' 'Cyclist' 'Cyclist' 'Cyclist'
 'Pedestrian' 'Cyclist' 'Car' 'Car' 'Cyclist' 'Pedestrian' 'Pedestrian'
 'Cyclist' 'Pedestrian' 'Cyclist' 'Pedestrian' 'Cyclist' 'Car' 'Car' 'Car'
 'Pedestrian' 'Car' 'Pedestrian' 'Pedestrian' 'Car' 'Pedestrian'
 'Pedestrian' 'Car' 'Car' 'Car' 'Car' 'Pedestrian' 'Car' 'Car' 'Car'
 'Cyclist' 'Car' 'Cyclist' 'Car' '

In [15]:
# get_label_annos with explicit file listing to debug
label_files = sorted(Path(GT_DIR).glob('*.txt'))
print(f"Found {len(label_files)} .txt files in GT_DIR")
if label_files:
    print("First file:", label_files[0])
    print("Contents:\n", label_files[0].read_text())

Found 81 .txt files in GT_DIR
First file: /OpenPCDet/datasets_converted/nuscenes/v1.0-mini_kitti_format/mini_val/label_2/048a45dd2cf54aa5808d8ccc85731d44.txt
Contents:
 car 0.00 0 -10.00 1563.28 469.81 1600.00 599.29 1.7 1.74 4.98 13.74 1.39 19.63 1.66 0.0000
motorcycle 0.00 0 -10.00 506.84 516.25 596.34 630.83 1.2 0.51 1.88 -3.15 1.55 15.25 -1.48 0.0000
motorcycle 0.00 0 -10.00 0.00 524.67 80.44 721.01 1.3 0.57 2.19 -6.92 1.62 10.18 -1.49 0.0000
motorcycle 0.00 0 -10.00 869.13 511.19 911.60 589.76 1.2 0.66 1.87 1.24 1.56 21.38 -1.54 0.0000
motorcycle 0.00 0 -10.00 224.09 486.50 383.04 626.97 1.5 0.72 2.61 -6.25 1.50 15.64 1.64 0.0000
car 0.00 0 -10.00 1147.04 497.72 1325.21 614.07 1.5 1.60 4.30 6.44 1.66 19.90 -1.51 0.0000
pedestrian 0.00 0 -10.00 449.40 451.52 569.37 655.61 1.8 0.73 0.82 -2.85 1.44 11.75 2.44 0.0000
motorcycle 0.00 0 -10.00 400.15 496.85 519.49 629.07 1.4 0.67 2.02 -4.19 1.50 15.05 -1.48 0.0000
car 0.00 0 -10.00 1427.62 493.07 1600.00 602.75 1.4 1.71 4.12 11.39 1.52 

## 5. Calculate KITTI Evaluation Metrics
To calculate the metrics locally, we need the Ground Truth labels in the exact same `.txt` format. Since you already ran the `export_kitti.py` converter earlier, the easiest and most accurate way is to use the `label_2` directory it generated (it handles tricky KITTI definitions like occlusion and truncation levels for easy/mod/hard splits).

We can load both your written prediction `.txt` directory and the Ground Truth `.txt` directory using OpenPCDet's native KITTI modules, and then run the evaluator!

In [16]:
from pcdet.datasets.kitti.kitti_object_eval_python import kitti_common
from pcdet.datasets.kitti.kitti_object_eval_python.eval import get_official_eval_result

# To get accurate Easy/Moderate/Hard splits, we MUST use the Ground Truth generated by `export_kitti.py`.
# nuScenes does not natively have occlusion/truncation. The ONLY way `export_kitti.py` splits difficulties
# is by projecting each 3D box into the 2D image plane and categorizing by pixel height (Easy >= 40px, Mod >= 25px).
# Using the pre-converted label_2 directory gives us those accurate 2D bounding box attributes.
GT_DIR = '/OpenPCDet/datasets_converted/nuscenes/v1.0-mini_kitti_format/mini_val/label_2'
PRED_DIR = str(OUTPUT_DIR)
LOG_FILE = Path('/OpenPCDet/tools/kitti_eval_results_log.txt')

# Deterministic matching by filename stem to avoid GT/pred misalignment.
gt_files = sorted(Path(GT_DIR).glob('*.txt'))
pred_files = sorted(Path(PRED_DIR).glob('*.txt'))
gt_stems = {f.stem: f for f in gt_files}
pred_stems = {f.stem: f for f in pred_files}
common = sorted(set(gt_stems) & set(pred_stems))

print(f"Matched pairs: {len(common)} / GT: {len(gt_stems)} / PRED: {len(pred_stems)}")
if not common:
    raise RuntimeError('No matched GT/PRED txt files found. Check directories and conversion step.')

print('Loading Ground Truth and Predictions in matched order...')
image_ids = [int(s) for s in common]
gt_annos = kitti_common.get_label_annos(GT_DIR, image_ids)
dt_annos = kitti_common.get_label_annos(PRED_DIR, image_ids)

print("\n--- Car Evaluation ---")
car_result = get_official_eval_result(gt_annos, dt_annos, 0)
print(car_result[0] if isinstance(car_result, tuple) else car_result)

with open(LOG_FILE, 'w') as f:
    f.write('--- Car Evaluation ---\n')
    f.write((car_result[0] if isinstance(car_result, tuple) else str(car_result)) + '\n')

    ped_result = get_official_eval_result(gt_annos, dt_annos, 1)
    f.write('\n--- Pedestrian ---\n')
    f.write((ped_result[0] if isinstance(ped_result, tuple) else str(ped_result)) + '\n')

    cyc_result = get_official_eval_result(gt_annos, dt_annos, 2)
    f.write('\n--- Cyclist ---\n')
    f.write((cyc_result[0] if isinstance(cyc_result, tuple) else str(cyc_result)) + '\n')

print(f"\nResults successfully saved to {LOG_FILE}")

Matched pairs: 0 / GT: 81 / PRED: 81


RuntimeError: No matched GT/PRED txt files found. Check directories and conversion step.